# Week 10 — Support Vector Machines & Kernel Methods
## Notebook 2: Hard-Margin SVM — The Primal Problem

| | |
|---|---|
| **Difficulty** | ⭐⭐⭐⭐ (Advanced) |
| **Estimated Time** | 3 hours |
| **Dataset Theme** | Email Spam Classification |
| **Key Skills** | Convex QP, KKT conditions, SVM primal, support vector verification |

---

## Why This Matters

Notebook 1 showed us *why* maximum margin is desirable. Now we need to answer: **how do we find it?**

The answer is the **Hard-Margin SVM** — a formal constrained optimization problem. Understanding this problem deeply will:
- Show you exactly what SVM optimizes (and why)
- Explain the KKT conditions and why only support vectors matter
- Reveal why SVMs produce **sparse** solutions
- Prepare you for the soft-margin and kernel extensions

---

## The Analogy First: The Town Planner's Road

Imagine you are a town planner. Your job is to draw a road exactly between two neighborhoods:

- **Spam Neighborhood** (red houses, top-right of the map)
- **Legitimate Neighborhood** (blue houses, bottom-left)

**Your hard rules:**
1. The road must be as **wide** as possible — you want a large buffer zone so there's no ambiguity about which neighborhood each house belongs to
2. **No house can be on the road itself** — every house must be clearly on one side or the other (no exceptions)

**Hard-margin means:** zero tolerance. No spam house can be on the legitimate side of the road, ever. This is only feasible if the two neighborhoods are completely separated to begin with — i.e., the data is linearly separable.

The **hard-margin SVM** finds the widest possible road subject to this strict no-violations constraint.

> If some houses are too close together (non-separable data), the hard-margin SVM simply **fails** — there is no feasible solution. That is why we need the soft-margin extension (Notebook 3).

## 1. The Optimization Problem: Plain English First

### What We Want

Find a hyperplane $\mathbf{w} \cdot \mathbf{x} + b = 0$ that:
1. **Correctly classifies every training point** (all spam on the spam side, all legitimate on the legitimate side)
2. **Maximizes the margin** $2 / \|\mathbf{w}\|$

### The Primal Formulation

We equivalently **minimize** $\|\mathbf{w}\|$ (which maximizes the margin):

$$\min_{\mathbf{w}, b} \quad \frac{1}{2} \|\mathbf{w}\|^2$$

$$\text{subject to} \quad y_i (\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1 \quad \forall i$$

**Why $\|\mathbf{w}\|^2$ instead of $2/\|\mathbf{w}\|$?**
- They are mathematically equivalent (one is maximized when the other is minimized)
- $\|\mathbf{w}\|^2$ is quadratic — smooth, differentiable, no square roots
- Quadratic objectives are easy for standard optimization solvers
- The $1/2$ is just a convenience to cancel factors of 2 in gradients

**What do the constraints mean?**
- $y_i = +1$ (spam): we need $\mathbf{w} \cdot \mathbf{x}_i + b \geq +1$
- $y_i = -1$ (legit): we need $\mathbf{w} \cdot \mathbf{x}_i + b \leq -1$
- Combined: $y_i (\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1$ for all $i$
- This ensures every point is on the correct side AND outside the margin

### The KKT Conditions (Intuition)

For constrained optimization, the **Karush-Kuhn-Tucker (KKT) conditions** characterize the solution:

- Each constraint $i$ gets a **Lagrange multiplier** $\alpha_i \geq 0$
- **Complementary slackness:** $\alpha_i \times [y_i(\mathbf{w} \cdot \mathbf{x}_i + b) - 1] = 0$

This means for every point, either:
- $\alpha_i = 0$: the point is NOT a support vector (it's away from the margin)
- $y_i(\mathbf{w} \cdot \mathbf{x}_i + b) = 1$: the point IS a support vector (it's on the margin)

**The weight vector from the dual solution:**
$$\mathbf{w} = \sum_i \alpha_i y_i \mathbf{x}_i$$

Since most $\alpha_i = 0$, the sum is over support vectors only. **This is why SVM solutions are sparse.**

In [ ]:
# ============================================================
# CELL 1: Imports and global settings
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize

# Reproducibility and consistent style
np.random.seed(42)
sns.set_theme(style='whitegrid')

plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 13

print("Libraries loaded successfully.")
print(f"NumPy   version: {np.__version__}")
print(f"Pandas  version: {pd.__version__}")
print(f"SciPy available: scipy.optimize.minimize imported")

## 2. Creating a Perfectly Separable Spam Dataset

For the hard-margin SVM, the data **must** be linearly separable — otherwise the constraints become infeasible and the optimization problem has no solution.

We use 100 emails (50 spam, 50 legitimate) with a clear gap between the classes.

In [ ]:
# ============================================================
# CELL 2: Generate perfectly linearly separable email dataset
# ============================================================
np.random.seed(42)

n_per_class = 50  # 50 spam + 50 legit = 100 emails

# Spam: high word count (320-480) + high spam word ratio (0.6-0.95)
X_spam = np.column_stack([
    np.random.uniform(320, 480, n_per_class),   # word_count
    np.random.uniform(0.60, 0.95, n_per_class)  # spam_word_ratio
])

# Legitimate: low word count (20-180) + low spam word ratio (0.05-0.40)
X_legit = np.column_stack([
    np.random.uniform(20, 180, n_per_class),    # word_count
    np.random.uniform(0.05, 0.40, n_per_class)  # spam_word_ratio
])

X_raw = np.vstack([X_spam, X_legit])         # raw features, shape (100, 2)
y     = np.hstack([np.ones(n_per_class),     # +1 = spam
                   -np.ones(n_per_class)])    # -1 = legitimate

# Scale features for numerical stability (critical for QP solvers)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)  # X is our working scaled feature matrix

# Tidy DataFrame for inspection
df = pd.DataFrame(X_raw, columns=['word_count', 'spam_word_ratio'])
df['label']    = y
df['class']    = df['label'].map({1.0: 'Spam', -1.0: 'Legitimate'})
df['x1_scaled'] = X[:, 0]
df['x2_scaled'] = X[:, 1]

spam_mask  = y ==  1
legit_mask = y == -1

print(f"Dataset shape  : {X.shape}")
print(f"Spam emails    : {spam_mask.sum()}")
print(f"Legit emails   : {legit_mask.sum()}")
print("\nFirst 5 rows (raw features):")
print(df[['word_count', 'spam_word_ratio', 'class']].head())

## 3. Implementing Hard-Margin SVM from Scratch

We solve the **primal** optimization problem directly using `scipy.optimize.minimize` with SLSQP (Sequential Least Squares Programming), which handles quadratic objectives with inequality constraints.

**The problem in standard form:**

$$\min_{\mathbf{w}, b} \quad f(\mathbf{w}) = \frac{1}{2}\mathbf{w}^\top \mathbf{w}$$

$$\text{s.t.} \quad g_i(\mathbf{w}, b) = y_i(\mathbf{w} \cdot \mathbf{x}_i + b) - 1 \geq 0 \quad \forall i = 1, \ldots, n$$

SLSQP requires us to supply the gradient of the objective: $\nabla_w f = \mathbf{w}$, $\partial f / \partial b = 0$.

In [ ]:
# ============================================================
# CELL 3: Hard-Margin SVM implemented from scratch via scipy QP
# ============================================================
def hard_margin_svm(X_data, y_labels):
    """
    Solve the hard-margin SVM primal optimization problem:

        minimize   (1/2) ||w||^2
        subject to yi * (w·xi + b) >= 1  for all i

    Uses scipy.optimize.minimize with the SLSQP method.

    Parameters
    ----------
    X_data    : array of shape (n_samples, n_features)
    y_labels  : array of shape (n_samples,) with values +1 / -1

    Returns
    -------
    w      : weight vector (n_features,)
    b      : bias scalar
    result : full scipy optimization result object
    """
    n, p = X_data.shape   # n = number of samples, p = number of features

    # The decision variable is params = [w1, w2, ..., wp, b]
    # length p+1 in total

    def objective(params):
        """Primal objective: (1/2)||w||^2  (ignore b — it doesn't affect the margin)."""
        w = params[:p]
        return 0.5 * np.dot(w, w)

    def objective_grad(params):
        """Gradient of objective w.r.t. params: [w1, w2, ..., wp, 0]."""
        w = params[:p]
        grad = np.zeros_like(params)
        grad[:p] = w   # d/dw (1/2 w^T w) = w
        grad[p]  = 0.0 # d/db (1/2 w^T w) = 0
        return grad

    # Build one inequality constraint per training point
    # SLSQP expects 'ineq' constraints of the form: fun(params) >= 0
    constraints = []
    for i in range(n):
        def constraint_i(params, idx=i):
            """yi * (w·xi + b) - 1 >= 0"""
            w = params[:p]
            b = params[p]
            return y_labels[idx] * (X_data[idx] @ w + b) - 1.0
        constraints.append({'type': 'ineq', 'fun': constraint_i})

    # Initial guess: zero vector (w=0, b=0)
    params0 = np.zeros(p + 1)

    result = minimize(
        objective,
        params0,
        jac=objective_grad,
        constraints=constraints,
        method='SLSQP',
        options={'ftol': 1e-9, 'maxiter': 2000, 'disp': False}
    )

    w_sol = result.x[:p]
    b_sol = result.x[p]
    return w_sol, b_sol, result


# Solve the hard-margin SVM
w_scratch, b_scratch, opt_result = hard_margin_svm(X, y)

margin_scratch = 2.0 / np.linalg.norm(w_scratch)

print("Hard-Margin SVM (from scratch — scipy SLSQP):")
print(f"  Optimizer status : {opt_result.message}")
print(f"  Converged        : {opt_result.success}")
print(f"  w                = {np.round(w_scratch, 4)}")
print(f"  b                = {b_scratch:.4f}")
print(f"  ||w||            = {np.linalg.norm(w_scratch):.4f}")
print(f"  Margin (2/||w||) = {margin_scratch:.4f}")

## 4. Verifying Against sklearn SVC

sklearn's `SVC(kernel='linear', C=1e6)` uses a highly optimized QP solver (LIBSVM) internally. When $C$ is very large, the soft-margin penalty becomes negligible and it behaves as a hard-margin SVM.

We compare our scratch solution to sklearn's to verify correctness.

In [ ]:
# ============================================================
# CELL 4: Fit sklearn SVC and compare to scratch solution
# ============================================================
# SVC with C=1e6 effectively enforces hard margin (no violations tolerated)
svm_sklearn = SVC(kernel='linear', C=1e6, random_state=42)
svm_sklearn.fit(X, y)

# Extract sklearn's learned parameters
w_sk = svm_sklearn.coef_[0]          # weight vector
b_sk = svm_sklearn.intercept_[0]     # bias
sv_idx_sk = svm_sklearn.support_     # support vector indices
n_sv_sk   = len(sv_idx_sk)
margin_sk = 2.0 / np.linalg.norm(w_sk)

# Compare the two solutions
print("Comparison: From-Scratch SLSQP vs sklearn SVC")
print("=" * 55)
print(f"{'Quantity':<25} {'Scratch':>15} {'sklearn':>15}")
print("-" * 55)
print(f"{'w[0]':<25} {w_scratch[0]:>15.4f} {w_sk[0]:>15.4f}")
print(f"{'w[1]':<25} {w_scratch[1]:>15.4f} {w_sk[1]:>15.4f}")
print(f"{'b':<25} {b_scratch:>15.4f} {b_sk:>15.4f}")
print(f"{'||w||':<25} {np.linalg.norm(w_scratch):>15.4f} {np.linalg.norm(w_sk):>15.4f}")
print(f"{'Margin (2/||w||)':<25} {margin_scratch:>15.4f} {margin_sk:>15.4f}")
print(f"{'Num support vectors':<25} {'N/A':>15} {n_sv_sk:>15d}")
print(f"{'Train accuracy':<25} {(np.sign(X @ w_scratch + b_scratch) == y).mean():>15.4f} "
      f"{svm_sklearn.score(X, y):>15.4f}")
print()

# Check direction agreement (w vectors may differ by scale; check if parallel)
cos_sim = np.dot(w_scratch, w_sk) / (np.linalg.norm(w_scratch) * np.linalg.norm(w_sk))
print(f"Cosine similarity between w vectors: {cos_sim:.6f}")
print(f"(Should be ≈ 1.0 if both point in the same direction)")
print(f"Margin agreement: |diff| = {abs(margin_scratch - margin_sk):.6f}")

In [ ]:
# ============================================================
# CELL 5: Comprehensive visualization of the Hard-Margin SVM
# Shows: decision boundary, margin lines, shaded margin,
# support vectors (highlighted with gold circles)
# ============================================================
def plot_svm(X_data, y_data, w, b, sv_indices_plot=None,
             title='Hard-Margin SVM', ax=None):
    """Helper function to plot SVM decision boundary and margins."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 7))

    w_norm = np.linalg.norm(w)
    margin_val = 2.0 / w_norm

    # Scatter plot
    ax.scatter(X_data[y_data ==  1, 0], X_data[y_data ==  1, 1],
               color='crimson', marker='o', s=60, alpha=0.65,
               edgecolors='darkred', linewidths=0.5,
               label='Spam (+1)', zorder=3)
    ax.scatter(X_data[y_data == -1, 0], X_data[y_data == -1, 1],
               color='steelblue', marker='s', s=60, alpha=0.65,
               edgecolors='navy', linewidths=0.5,
               label='Legitimate (-1)', zorder=3)

    # Decision boundary and margin lines
    x1_range = np.linspace(X_data[:, 0].min() - 0.4, X_data[:, 0].max() + 0.4, 300)
    # Decision boundary: w[0]*x1 + w[1]*x2 + b = 0
    x2_db  = (-w[0] * x1_range - b) / w[1]
    x2_pos = (-w[0] * x1_range - b + 1) / w[1]  # w·x+b = +1
    x2_neg = (-w[0] * x1_range - b - 1) / w[1]  # w·x+b = -1

    ax.plot(x1_range, x2_db,  color='black', lw=2.5,
            label='Decision Boundary (w·x+b=0)')
    ax.plot(x1_range, x2_pos, color='forestgreen', lw=1.8, ls='--',
            label='Margin: w·x+b=+1 (spam side)')
    ax.plot(x1_range, x2_neg, color='forestgreen', lw=1.8, ls='--',
            label='Margin: w·x+b=−1 (legit side)')
    ax.fill_between(x1_range, x2_neg, x2_pos,
                    color='limegreen', alpha=0.15, label='Margin region')

    # Highlight support vectors if provided
    if sv_indices_plot is not None:
        ax.scatter(X_data[sv_indices_plot, 0], X_data[sv_indices_plot, 1],
                   s=280, facecolors='none', edgecolors='gold',
                   linewidths=2.8, zorder=5, label='Support Vectors')

    ax.set_xlim(X_data[:, 0].min() - 0.4, X_data[:, 0].max() + 0.4)
    ax.set_xlabel('Word Count (scaled)')
    ax.set_ylabel('Spam Word Ratio (scaled)')
    ax.set_title(f'{title}\nMargin = {margin_val:.4f}  |  SVs = {len(sv_indices_plot) if sv_indices_plot is not None else "?"}')
    ax.legend(fontsize=9, loc='upper left')
    return ax


# Plot using sklearn's solution (more numerically accurate w and support vectors)
fig, ax = plt.subplots(figsize=(11, 8))
plot_svm(X, y, w_sk, b_sk, sv_indices_plot=sv_idx_sk,
         title='Hard-Margin SVM — Email Spam Classification\n(sklearn SVC, C=1e6)', ax=ax)
plt.tight_layout()
plt.show()

print(f"Number of support vectors: {n_sv_sk}")
print(f"Support vector indices   : {sv_idx_sk.tolist()}")

## 5. Primal Feasibility Check

We verify that the solution satisfies **all** primal constraints:

$$y_i (\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1 \quad \forall i$$

This is a direct sanity check that the optimization actually found a feasible solution.

In [ ]:
# ============================================================
# CELL 6: Primal constraint verification
# All functional margins must be >= 1
# ============================================================
# Compute functional margins using sklearn's w and b
functional_margins = y * (X @ w_sk + b_sk)

# Tolerance for floating point
tol = 1e-4
all_satisfied = (functional_margins >= 1.0 - tol).all()

print("Primal Constraint Verification: yi * (w·xi + b) >= 1")
print("=" * 55)
print(f"  Total constraints   : {len(functional_margins)}")
print(f"  All satisfied (tol={tol}): {all_satisfied}")
print(f"  Min functional margin  : {functional_margins.min():.6f} (should be ≥ 1)")
print(f"  Max functional margin  : {functional_margins.max():.6f}")
print(f"  Mean functional margin : {functional_margins.mean():.6f}")
print()

# Show constraints for support vectors (should be exactly = 1)
print("Functional margins for support vectors (should be ≈ 1.0):")
for idx in sv_idx_sk:
    class_name = 'Spam (+1)' if y[idx] == 1 else 'Legit (-1)'
    print(f"  Point {idx:3d} [{class_name}]  "
          f"y*(w·x+b) = {functional_margins[idx]:.6f}")

# Check non-support-vectors are strictly inside margin
non_sv_mask = np.ones(len(X), dtype=bool)
non_sv_mask[sv_idx_sk] = False
print(f"\nNon-SV functional margins (all should be > 1):")
print(f"  Min = {functional_margins[non_sv_mask].min():.4f}")
print(f"  All > 1: {(functional_margins[non_sv_mask] > 1.0 - tol).all()}")

In [ ]:
# ============================================================
# CELL 7: Visualize the functional margin distribution
# Support vectors at margin=1; non-SVs strictly above
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: histogram of functional margins by class
ax = axes[0]
ax.hist(functional_margins[spam_mask], bins=18, color='crimson',
        alpha=0.6, edgecolor='darkred', label='Spam (+1)')
ax.hist(functional_margins[legit_mask], bins=18, color='steelblue',
        alpha=0.6, edgecolor='navy', label='Legitimate (-1)')
ax.axvline(1.0, color='forestgreen', lw=2, ls='--',
           label='Margin boundary (= 1)')
ax.set_xlabel('Functional Margin  yᵢ(w·xᵢ + b)')
ax.set_ylabel('Count')
ax.set_title('Functional Margin Distribution\nAll points must be ≥ 1')
ax.legend(fontsize=10)

# Right: sorted functional margins for all points
ax2 = axes[1]
sorted_idx = np.argsort(functional_margins)
colors_bar = ['crimson' if y[i] == 1 else 'steelblue' for i in sorted_idx]

# Mark support vectors with a special edge color
sv_set = set(sv_idx_sk.tolist())
edge_colors = ['gold' if i in sv_set else 'gray' for i in sorted_idx]
edge_widths = [2.0 if i in sv_set else 0.3 for i in sorted_idx]

bars = ax2.bar(range(len(sorted_idx)), functional_margins[sorted_idx],
               color=colors_bar, alpha=0.75,
               edgecolor=edge_colors, linewidth=edge_widths)
ax2.axhline(1.0, color='forestgreen', lw=2.0, ls='--',
            label='Margin = 1 (support vectors)')

# Add legend entries for colors
legend_elements = [
    mpatches.Patch(facecolor='crimson', label='Spam emails'),
    mpatches.Patch(facecolor='steelblue', label='Legitimate emails'),
    mpatches.Patch(facecolor='white', edgecolor='gold', lw=2, label='Support Vectors'),
    Line2D([0], [0], color='forestgreen', lw=2, ls='--', label='Margin = 1'),
]
ax2.set_xlabel('Emails (sorted by functional margin)')
ax2.set_ylabel('Functional Margin')
ax2.set_title('All Functional Margins — Sorted\nGold borders = support vectors')
ax2.legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
plt.show()

## 6. The KKT Sparsity Property: w = Σ αᵢ yᵢ xᵢ

From the KKT conditions of the dual problem, the weight vector can be written as:

$$\mathbf{w} = \sum_{i=1}^{n} \alpha_i y_i \mathbf{x}_i$$

where $\alpha_i \geq 0$ are the dual variables (Lagrange multipliers). The key insight from complementary slackness:

$$\alpha_i > 0 \quad \Leftrightarrow \quad y_i(\mathbf{w} \cdot \mathbf{x}_i + b) = 1 \quad \Leftrightarrow \quad \mathbf{x}_i \text{ is a support vector}$$

In sklearn, the dual coefficients are stored as `dual_coef_` = $\alpha_i y_i$, and support vectors as `support_vectors_`.

In [ ]:
# ============================================================
# CELL 8: Verify w = Σ αi * yi * xi using sklearn dual variables
# ============================================================
# sklearn stores dual_coef_ = alpha_i * y_i for each support vector
# So w = sum over SVs of dual_coef_[i] * support_vector[i]

dual_coef    = svm_sklearn.dual_coef_[0]          # shape: (n_sv,)  = alpha_i * y_i
support_vecs = svm_sklearn.support_vectors_        # shape: (n_sv, p) = the SV feature vectors

# Reconstruct w from dual coefficients and support vectors
# w = sum_i (alpha_i * y_i) * x_i  = dual_coef @ support_vectors
w_from_dual = dual_coef @ support_vecs             # matrix multiply: (1,n_sv) x (n_sv,p) = (1,p)

print("Verifying: w = Σ αᵢ yᵢ xᵢ (sum only over support vectors)")
print("=" * 60)
print(f"  Number of support vectors         : {len(dual_coef)}")
print(f"  Dual coefficients (αᵢ × yᵢ)      : {np.round(dual_coef, 4)}")
print()
print(f"  w from sklearn (coef_)            : {np.round(w_sk, 6)}")
print(f"  w reconstructed from dual         : {np.round(w_from_dual, 6)}")
print(f"  Max absolute difference           : {np.abs(w_sk - w_from_dual).max():.2e}")
print(f"  Reconstruction successful         : {np.allclose(w_sk, w_from_dual, atol=1e-4)}")
print()
print("Key insight: w is a LINEAR COMBINATION of only the support vectors.")
print("All 90+ non-support-vector emails contribute ZERO to w.")
print("→ This is the sparsity of SVMs.")

In [ ]:
# ============================================================
# CELL 9: Visualize the dual coefficient contribution
# Show how each support vector contributes to w
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: alpha_i * y_i for each support vector
ax = axes[0]
sv_labels = [f'SV {i}\n({"Spam" if y[idx]==1 else "Legit"})'
             for i, idx in enumerate(sv_idx_sk)]
bar_colors = ['crimson' if y[idx] == 1 else 'steelblue' for idx in sv_idx_sk]

ax.bar(sv_labels, dual_coef, color=bar_colors, alpha=0.8,
       edgecolor='black', linewidth=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Support Vector')
ax.set_ylabel('Dual coefficient (αᵢ × yᵢ)')
ax.set_title('Dual Coefficients per Support Vector\n'
             'Positive = spam SV, Negative = legit SV')

# Right: show w = sum of contributions
ax2 = axes[1]
# Each contribution is a 2D vector: dual_coef[i] * support_vecs[i]
contributions = np.array([dc * sv for dc, sv in zip(dual_coef, support_vecs)])

x_coords = np.arange(len(sv_idx_sk))
width = 0.35
ax2.bar(x_coords - width/2, contributions[:, 0], width=width,
        color='cornflowerblue', edgecolor='navy', alpha=0.8, label='w[0] contribution')
ax2.bar(x_coords + width/2, contributions[:, 1], width=width,
        color='salmon', edgecolor='darkred', alpha=0.8, label='w[1] contribution')
ax2.axhline(0, color='black', lw=0.8)

# Annotate with the sum
ax2.axhline(w_sk[0], color='cornflowerblue', lw=2, ls='--',
            label=f'Total w[0] = {w_sk[0]:.3f}')
ax2.axhline(w_sk[1], color='salmon', lw=2, ls='--',
            label=f'Total w[1] = {w_sk[1]:.3f}')

ax2.set_xticks(x_coords)
ax2.set_xticklabels(sv_labels)
ax2.set_xlabel('Support Vector')
ax2.set_ylabel('Contribution to w')
ax2.set_title('Contributions to w from Each Support Vector\n'
              'w = Σ αᵢ yᵢ xᵢ — summing all bars gives the final w')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 7. Experiment: Removing Non-Support-Vectors

A remarkable property of SVMs: **the decision boundary is completely determined by the support vectors**.

If we delete all non-support-vector points from the training set and refit, we should get the **exact same** decision boundary and margin.

This experiment makes the sparsity property concrete and demonstrates why SVMs are memory-efficient at prediction time.

In [ ]:
# ============================================================
# CELL 10: Remove non-SVs and refit — boundary should be unchanged
# ============================================================
# Identify non-support-vector indices
non_sv_idx = np.array([i for i in range(len(X)) if i not in sv_set])
n_to_remove = min(10, len(non_sv_idx))  # remove 10 non-SVs

np.random.seed(42)
remove_idx = np.random.choice(non_sv_idx, size=n_to_remove, replace=False)
keep_mask  = np.ones(len(X), dtype=bool)
keep_mask[remove_idx] = False

X_reduced = X[keep_mask]
y_reduced = y[keep_mask]

print(f"Original training set: {len(X)} points")
print(f"After removing {n_to_remove} non-SVs: {len(X_reduced)} points")
print(f"Points removed (non-SVs): {remove_idx.tolist()}")

# Refit on reduced set
svm_reduced = SVC(kernel='linear', C=1e6, random_state=42)
svm_reduced.fit(X_reduced, y_reduced)

w_red = svm_reduced.coef_[0]
b_red = svm_reduced.intercept_[0]
margin_red = 2.0 / np.linalg.norm(w_red)

print()
print("Comparing original vs. reduced dataset SVM:")
print(f"{'Quantity':<30} {'Original':>12} {'Reduced':>12}")
print("-" * 56)
print(f"{'w[0]':<30} {w_sk[0]:>12.6f} {w_red[0]:>12.6f}")
print(f"{'w[1]':<30} {w_sk[1]:>12.6f} {w_red[1]:>12.6f}")
print(f"{'b':<30} {b_sk:>12.6f} {b_red:>12.6f}")
print(f"{'Margin (2/||w||)':<30} {margin_sk:>12.6f} {margin_red:>12.6f}")
print(f"{'Num SVs':<30} {n_sv_sk:>12d} {len(svm_reduced.support_):>12d}")
print()
print(f"Boundaries match: {np.allclose(w_sk, w_red, atol=1e-3) and np.isclose(b_sk, b_red, atol=1e-3)}")
print("→ Removing non-SVs does NOT change the decision boundary.")

In [ ]:
# ============================================================
# CELL 11: Side-by-side plot — original vs. reduced dataset
# Shows that the boundary is identical even after removing non-SVs
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plot_svm(X, y, w_sk, b_sk, sv_indices_plot=sv_idx_sk,
         title=f'Original Dataset ({len(X)} points)', ax=axes[0])
axes[0].set_title(f'Original Dataset ({len(X)} points)\n'
                  f'Margin = {margin_sk:.4f}  |  {n_sv_sk} SVs')

# For reduced plot: mark the removed points
plot_svm(X_reduced, y_reduced, w_red, b_red,
         sv_indices_plot=svm_reduced.support_,
         title=f'Reduced Dataset ({len(X_reduced)} points — {n_to_remove} non-SVs removed)',
         ax=axes[1])
axes[1].set_title(f'Reduced Dataset ({len(X_reduced)} points — {n_to_remove} non-SVs removed)\n'
                  f'Margin = {margin_red:.4f}  |  {len(svm_reduced.support_)} SVs  '
                  f'(SAME boundary!)')

plt.tight_layout()
plt.show()
print("Observation: The decision boundaries are identical — removing non-SVs has no effect.")

## 8. Experiment: Moving a Support Vector vs. a Non-Support-Vector

This experiment demonstrates the asymmetric role of support vectors:

- **Moving a non-support-vector**: the boundary does not change
- **Moving a support vector**: the boundary shifts noticeably

This is the most vivid demonstration of why support vectors are "support" vectors — they literally hold the boundary in place.

In [ ]:
# ============================================================
# CELL 12: Move a support vector — show boundary shifts
# Move a non-support-vector — show boundary unchanged
# ============================================================
shift_amount = 0.35  # how far to move the point in scaled space

# ---- Experiment A: Move a non-support-vector ----
non_sv_to_move = non_sv_idx[0]  # pick first non-SV

X_moved_nonsv = X.copy()
# Move it closer to the boundary (in the direction of the boundary)
X_moved_nonsv[non_sv_to_move, 0] += shift_amount
X_moved_nonsv[non_sv_to_move, 1] += shift_amount

svm_nonsv_moved = SVC(kernel='linear', C=1e6, random_state=42)
svm_nonsv_moved.fit(X_moved_nonsv, y)
w_nonsv = svm_nonsv_moved.coef_[0]
b_nonsv = svm_nonsv_moved.intercept_[0]

# ---- Experiment B: Move a support vector ----
sv_to_move = sv_idx_sk[0]  # pick first SV

X_moved_sv = X.copy()
# Move it away from the boundary (perpendicular to boundary normal)
X_moved_sv[sv_to_move, 0] += shift_amount
X_moved_sv[sv_to_move, 1] += shift_amount

svm_sv_moved = SVC(kernel='linear', C=1e6, random_state=42)
svm_sv_moved.fit(X_moved_sv, y)
w_sv = svm_sv_moved.coef_[0]
b_sv = svm_sv_moved.intercept_[0]

# Report changes
w_change_nonsv = np.linalg.norm(w_nonsv - w_sk)
w_change_sv    = np.linalg.norm(w_sv    - w_sk)
margin_nonsv   = 2.0 / np.linalg.norm(w_nonsv)
margin_sv_new  = 2.0 / np.linalg.norm(w_sv)

print("Effect of Moving Points on the Decision Boundary")
print("=" * 60)
print(f"{'Experiment':<35} {'||Δw||':>10} {'Margin':>10}")
print("-" * 60)
print(f"{'Original (baseline)':<35} {'—':>10} {margin_sk:>10.4f}")
print(f"{'Move non-SV (point {non_sv_to_move})':<35} {w_change_nonsv:>10.4f} {margin_nonsv:>10.4f}")
print(f"{'Move SV (point {sv_to_move})':<35} {w_change_sv:>10.4f} {margin_sv_new:>10.4f}")
print()
print(f"Moving non-SV changes ||w|| by: {w_change_nonsv:.6f} (≈ 0 expected)")
print(f"Moving SV    changes ||w|| by: {w_change_sv:.6f} (> 0 expected)")

In [ ]:
# ============================================================
# CELL 13: Visualize effect of moving points on the boundary
# Left: moving non-SV (no change); Right: moving SV (boundary shifts)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Common x1 range
x1_range = np.linspace(X[:, 0].min() - 0.4, X[:, 0].max() + 0.4, 300)

def get_line(w, b, shift):
    """Compute x2 for decision boundary or margin lines."""
    return (-w[0] * x1_range - b + shift) / w[1]

for ax, (w_new, b_new, X_data, moved_idx, label, color_new) in zip(
    axes,
    [
        (w_nonsv, b_nonsv, X_moved_nonsv, non_sv_to_move,
         f'Non-SV moved (point {non_sv_to_move})\n→ Boundary UNCHANGED', 'purple'),
        (w_sv,    b_sv,    X_moved_sv,    sv_to_move,
         f'SV moved (point {sv_to_move})\n→ Boundary SHIFTS', 'orange'),
    ]
):
    # Original data (with original boundary in black)
    ax.scatter(X[y ==  1, 0], X[y ==  1, 1], color='crimson', marker='o',
               s=50, alpha=0.4, edgecolors='darkred', linewidths=0.4)
    ax.scatter(X[y == -1, 0], X[y == -1, 1], color='steelblue', marker='s',
               s=50, alpha=0.4, edgecolors='navy', linewidths=0.4)

    # Highlight the moved point
    ax.scatter(*X_data[moved_idx], s=250, color=color_new,
               edgecolors='black', linewidths=1.5, zorder=6,
               label=f'Moved point ({moved_idx})')

    # Original boundary (dashed, lighter)
    ax.plot(x1_range, get_line(w_sk, b_sk, 0), color='gray',
            lw=1.5, ls=':', label='Original boundary', zorder=3)

    # New boundary (solid, darker)
    ax.plot(x1_range, get_line(w_new, b_new, 0), color='black',
            lw=2.5, label='New boundary', zorder=4)
    ax.plot(x1_range, get_line(w_new, b_new, +1), color='forestgreen',
            lw=1.5, ls='--', label='New margin lines')
    ax.plot(x1_range, get_line(w_new, b_new, -1), color='forestgreen',
            lw=1.5, ls='--')

    # Highlight original SVs
    ax.scatter(X[sv_idx_sk, 0], X[sv_idx_sk, 1], s=220, facecolors='none',
               edgecolors='gold', linewidths=2.5, zorder=5, label='Original SVs')

    ax.set_xlim(X[:, 0].min() - 0.4, X[:, 0].max() + 0.4)
    ax.set_xlabel('Word Count (scaled)')
    ax.set_ylabel('Spam Word Ratio (scaled)')
    new_margin_val = 2.0 / np.linalg.norm(w_new)
    ax.set_title(f'{label}\nNew margin = {new_margin_val:.4f}  (orig = {margin_sk:.4f})')
    ax.legend(fontsize=8, loc='upper left')

plt.suptitle('Support Vector Role: Moving a Point\nLeft = Non-SV (no effect)   Right = SV (boundary shifts)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 9. Limitations of Hard-Margin SVM

The hard-margin SVM is elegant, but has a critical limitation: it **requires perfectly linearly separable data**.

What happens when data is not separable?
- The constraint $y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1$ for ALL $i$ becomes **infeasible**
- The optimizer cannot find any solution
- The algorithm fails

We demonstrate this by adding a small amount of noise to make the data non-separable.

In [ ]:
# ============================================================
# CELL 14: Demonstrate hard-margin SVM failure on overlapping data
# ============================================================
np.random.seed(42)

# Create a non-separable dataset by overlapping the class regions
X_spam_ns = np.column_stack([
    np.random.uniform(200, 400, 50),   # overlapping word count range
    np.random.uniform(0.35, 0.75, 50)  # overlapping spam word ratio
])
X_legit_ns = np.column_stack([
    np.random.uniform(100, 300, 50),
    np.random.uniform(0.25, 0.65, 50)
])

X_ns_raw = np.vstack([X_spam_ns, X_legit_ns])
y_ns     = np.hstack([np.ones(50), -np.ones(50)])
X_ns     = StandardScaler().fit_transform(X_ns_raw)

# Attempt hard-margin SVM (very large C = hard margin behavior)
print("Attempting hard-margin SVM on overlapping (non-separable) data...")
try:
    svm_hard_ns = SVC(kernel='linear', C=1e9, random_state=42)
    svm_hard_ns.fit(X_ns, y_ns)
    acc_hard = svm_hard_ns.score(X_ns, y_ns)
    print(f"Hard-margin SVM training accuracy: {acc_hard:.4f}")
    if acc_hard < 1.0:
        print("  → Training accuracy < 1.0: data is NOT linearly separable.")
        print("  → Hard-margin assumption violated: some points inside the margin.")
except Exception as e:
    print(f"Error: {e}")

# Check our from-scratch implementation
print("\nAttempting scipy SLSQP hard-margin on overlapping data...")
w_fail, b_fail, result_fail = hard_margin_svm(X_ns, y_ns)
print(f"  Optimizer status  : {result_fail.message}")
print(f"  Converged         : {result_fail.success}")
print(f"  Objective value   : {result_fail.fun:.4f}")
print()
# Check how many constraints are violated
fm_fail = y_ns * (X_ns @ w_fail + b_fail)
violated = (fm_fail < 1.0 - 1e-3).sum()
print(f"  Constraints violated: {violated} / {len(X_ns)}")
print("  → Soft-margin SVM (Notebook 3) is needed to handle non-separable data.")

In [ ]:
# ============================================================
# CELL 15: Summary table of all key results
# ============================================================
summary = pd.DataFrame({
    'Property': [
        'Primal objective',
        'Constraints',
        'Solution type',
        'Weight vector w',
        'Bias b',
        'Norm ||w||',
        'Margin 2/||w||',
        'Number of support vectors',
        'w reconstructed from dual',
        'Training accuracy',
        'Requires linearly separable?',
    ],
    'Value / Formula': [
        'min (1/2)||w||²',
        'yᵢ(w·xᵢ + b) ≥ 1 ∀i',
        'Convex QP — unique global minimum',
        str(np.round(w_sk, 4)),
        f'{b_sk:.4f}',
        f'{np.linalg.norm(w_sk):.4f}',
        f'{margin_sk:.4f}',
        str(n_sv_sk),
        f'{np.allclose(w_sk, w_from_dual, atol=1e-4)} (verified)',
        f'{svm_sklearn.score(X, y):.4f}',
        'YES — fails if data overlaps',
    ]
})

print("Hard-Margin SVM — Complete Results Summary")
print("=" * 65)
print(summary.to_string(index=False))

## 10. Key Takeaways

| Concept | Summary |
|---|---|
| **Primal problem** | Minimize $\frac{1}{2}\|\mathbf{w}\|^2$ subject to $y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1$ |
| **Convex QP** | Unique global minimum — guaranteed to find the optimal solution |
| **Constraints** | Each training point must be on the correct side with margin ≥ 1 |
| **KKT / Complementary slackness** | $\alpha_i > 0$ only for support vectors; all others $\alpha_i = 0$ |
| **Sparsity** | $\mathbf{w} = \sum_i \alpha_i y_i \mathbf{x}_i$ — sum over SVs only |
| **Removing non-SVs** | Does NOT change the boundary or margin |
| **Moving an SV** | Changes the boundary; moving a non-SV does not |
| **Hard-margin limitation** | Requires perfectly linearly separable data — fails otherwise |

**What comes next:**
- **Notebook 3: Soft-Margin SVM** — introduce slack variables $\xi_i$ to allow some misclassifications; add penalty $C \sum \xi_i$ to the objective
- **Notebook 4: Kernel SVM** — replace $\mathbf{x}_i \cdot \mathbf{x}_j$ with $K(\mathbf{x}_i, \mathbf{x}_j)$ to handle non-linear boundaries

---

## Self-Check Questions

### Question 1
The hard-margin SVM **fails** if data is not linearly separable. Why can't it just find the "best" separating hyperplane even when perfect separation is impossible?

<details>
<summary>Click to reveal answer</summary>

The hard-margin SVM is formulated as a **constrained optimization problem**:

$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 \quad \text{s.t.} \quad y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1 \quad \forall i$$

If the data is **not** linearly separable, there is no $\mathbf{w}$ and $b$ that simultaneously satisfies ALL of the constraints $y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1$. At least one point on the wrong side will violate its constraint no matter what hyperplane you choose.

The problem becomes **infeasible** — the feasible set (the set of $(\mathbf{w}, b)$ that satisfies all constraints) is **empty**. There is simply no point to minimize over, so the optimizer has nothing to return.

In practice, optimizers like SLSQP will either fail to converge, return a poor solution, or report that the problem is infeasible. The solution is to **relax the constraints** using slack variables $\xi_i$:

$$y_i(\mathbf{w} \cdot \mathbf{x}_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0$$

This is the soft-margin SVM (Notebook 3), which can always find a solution by paying a penalty $C \sum \xi_i$ for violations.

</details>

### Question 2
After fitting, $\mathbf{w} = \sum_i \alpha_i y_i \mathbf{x}_i$ and only support vectors have $\alpha_i > 0$. Why does this mean the SVM solution is **"sparse"**?

<details>
<summary>Click to reveal answer</summary>

**Sparsity** means that the solution depends on only a small subset of the training data — specifically, only the support vectors.

From the KKT complementary slackness condition:

$$\alpha_i \times [y_i(\mathbf{w} \cdot \mathbf{x}_i + b) - 1] = 0$$

For non-support-vectors, $y_i(\mathbf{w} \cdot \mathbf{x}_i + b) > 1$ (they are strictly inside the correct margin region), so the bracket $[\cdot]$ is **nonzero**. For the product to equal zero, we must have $\alpha_i = 0$.

Since $\mathbf{w} = \sum_i \alpha_i y_i \mathbf{x}_i$, and $\alpha_i = 0$ for all non-SVs, those terms contribute **nothing** to $\mathbf{w}$. The sum effectively runs over only the support vectors.

**Why is sparsity useful?**
1. **Memory efficiency**: you only need to store the support vectors, not the entire training set
2. **Prediction speed**: the decision function $f(\mathbf{x}) = \text{sign}(\sum_{i \in SV} \alpha_i y_i K(\mathbf{x}_i, \mathbf{x}) + b)$ only involves the support vectors
3. **Interpretability**: the support vectors are the "critical" examples that define the boundary
4. **Mathematical elegance**: a model with 1000 training points but 5 SVs is essentially a 5-point model

This contrasts sharply with models like k-NN, which require storing and searching all training points at prediction time.

</details>

### Question 3
You have 1000 training points but only 5 support vectors. If you delete 990 non-support-vector points, does the model change? What does this tell you about SVMs vs. models like k-NN?

<details>
<summary>Click to reveal answer</summary>

**No, the model does NOT change.** The decision boundary, margin, and support vectors would be exactly the same whether you train on all 1000 points or just the 5 support vectors (plus perhaps a few more for stability in the optimizer).

**Why?** The SVM objective and constraints are determined only by the support vectors — the 995 non-SV points have $\alpha_i = 0$ and therefore contribute $\mathbf{0}$ to $\mathbf{w}$. They are "invisible" to the model at the margin level.

**Comparison with k-NN (k-Nearest Neighbors):**

| Property | SVM | k-NN |
|---|---|---|
| Training memory | Only support vectors needed | All training points needed |
| Prediction cost | O(n_sv) — number of SVs | O(n) — ALL training points |
| If you delete non-SVs | Model unchanged | Model changes significantly |
| What defines the boundary | Support vectors only | Nearest neighbors to test point |
| Sensitivity to far-away points | None | None (k-NN ignores far points) |
| Sensitivity to noisy outliers | Only if outlier becomes SV | High — outliers can be neighbors |

**The big picture:** SVMs learn a **global model** (the hyperplane) that is defined by a **local, sparse** set of training points (the SVs). k-NN, by contrast, is a **local, dense** method that uses all training points at prediction time and has no global model.

This is why SVMs can scale to large feature spaces efficiently (especially with kernels), while k-NN becomes slow as $n$ grows.

</details>